# API wrappers

The OpenWeatherMap offers REST endpoints for querying current weather, forecasts, historical data, etc. However, accessing this data directly via the REST API requires handling multiple API calls, query parameters, and response parsing. The pyowm library abstracts these complexities and provides useful built-in functionalities.

After signing in to OpenWeatherMap retrieve your api key at https://home.openweathermap.org/api_keys

You will also need to install the pyowm package: pip install pyowm

In [4]:

!pip install pyowm

import requests
import pyowm
import json

from getpass import getpass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 39.3 MB/s eta 0:00:00


In [5]:
api_key = getpass("OpenWeatherMap API key:")

OpenWeatherMap API key:··········


## use case 1: managing API keys

In a raw rest API call you always have to manage credentials in each individual call. Wrappers usually store and manage the authentication for you

In [6]:
#You can get current weather data by making a GET request to an endpoint like:

params = {
    'appid' : api_key
}

response = requests.get('https://api.openweathermap.org/data/2.5/weather?q=London', params = params)

json.loads(response.text)

#but for every call you make using GET from now on you do need to add the parameters, since the raw API does not manage authentication for you

{'coord': {'lon': -0.1257, 'lat': 51.5085},
 'weather': [{'id': 803,
   'main': 'Clouds',
   'description': 'broken clouds',
   'icon': '04d'}],
 'base': 'stations',
 'main': {'temp': 281.07,
  'feels_like': 280.38,
  'temp_min': 279.77,
  'temp_max': 282.04,
  'pressure': 1025,
  'humidity': 89,
  'sea_level': 1025,
  'grnd_level': 1021},
 'visibility': 10000,
 'wind': {'speed': 1.54, 'deg': 350},
 'clouds': {'all': 51},
 'dt': 1772526263,
 'sys': {'type': 2,
  'id': 2075535,
  'country': 'GB',
  'sunrise': 1772520119,
  'sunset': 1772559799},
 'timezone': 0,
 'id': 2643743,
 'name': 'London',
 'cod': 200}

Most wrappers (pyowm included) include some way of initializing a session with the authentication key that you then don't need to type again.

Initialize pyowm with the default configuration. Thenopen the weather manager

Check out a snippet here: https://pyowm.readthedocs.io/en/latest/v3/code-recipes.html#weather_data

In [7]:
from pyowm import OWM
owm = OWM(api_key)
mgr = owm.weather_manager()

obs = mgr.weather_at_place("London,GB")
w = obs.weather

w.temperature("celsius"), w.status, w.detailed_status

({'temp': 8.18,
  'temp_max': 9.05,
  'temp_min': 6.68,
  'feels_like': 7.53,
  'temp_kf': None},
 'Clouds',
 'broken clouds')

## use case 2: Simplified calls

With the raw REST API, you'd have to build a URL manually, send the request, and parse the JSON response to get the current weather.

Get the equivalent call as above for the city of London using the pyowm package

In [9]:
#your code herecity = 'London'
url = f'http://api.openweathermap.org/data/2.5/weather?q={city}'

response = requests.get(url,params= params)
data = response.json()
temperature = data['main']['temp']
humidity = data['main']['humidity']
wind_speed = data['wind']['speed']

print(f"Temperature: {temperature}°C, Humidity: {humidity}%, Wind Speed: {wind_speed} m/s")

Temperature: 281.33°C, Humidity: 89%, Wind Speed: 1.54 m/s


## use case 3: Combining and chaining calls

Wrappers often offer methods that make multiple calls to batch requests that make sense to batch. And often they offer methods that make sequences of calls that each returns information necessary to make the next call.

For example, to get a weather forecast for a specific city using the raw API you need to first geocode the city to get its latitude and longitude:

In [10]:
city = 'New York'
geocode_url = f'http://api.openweathermap.org/data/2.5/weather?q={city}'
geocode_response = requests.get(geocode_url,params=params).json()

lat = geocode_response['coord']['lat']
lon = geocode_response['coord']['lon']

Then, request the weather forecast for that latitude/longitude:

In [11]:
forecast_url = f'http://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}'
forecast_response = requests.get(forecast_url, params=params).json()

for entry in forecast_response['list']:
    print(f"Time: {entry['dt_txt']}, Temp: {entry['main']['temp']}°C")

Time: 2026-03-03 09:00:00, Temp: 272.75°C
Time: 2026-03-03 12:00:00, Temp: 272.96°C
Time: 2026-03-03 15:00:00, Temp: 273.76°C
Time: 2026-03-03 18:00:00, Temp: 274.24°C
Time: 2026-03-03 21:00:00, Temp: 274.36°C
Time: 2026-03-04 00:00:00, Temp: 274.53°C
Time: 2026-03-04 03:00:00, Temp: 274.98°C
Time: 2026-03-04 06:00:00, Temp: 275.32°C
Time: 2026-03-04 09:00:00, Temp: 275.59°C
Time: 2026-03-04 12:00:00, Temp: 276.15°C
Time: 2026-03-04 15:00:00, Temp: 277.85°C
Time: 2026-03-04 18:00:00, Temp: 279.14°C
Time: 2026-03-04 21:00:00, Temp: 278.76°C
Time: 2026-03-05 00:00:00, Temp: 277.59°C
Time: 2026-03-05 03:00:00, Temp: 277.05°C
Time: 2026-03-05 06:00:00, Temp: 276.97°C
Time: 2026-03-05 09:00:00, Temp: 277.12°C
Time: 2026-03-05 12:00:00, Temp: 277.15°C
Time: 2026-03-05 15:00:00, Temp: 277.17°C
Time: 2026-03-05 18:00:00, Temp: 277.34°C
Time: 2026-03-05 21:00:00, Temp: 277.95°C
Time: 2026-03-06 00:00:00, Temp: 277.36°C
Time: 2026-03-06 03:00:00, Temp: 277.63°C
Time: 2026-03-06 06:00:00, Temp: 2

Two calls: one for geocoding, one for forecasts.
But with pyowm, because this is a common operation, there is a method that handles the geocoding internally and then fetches the weather forecast in one step.

Get the above forecast in a single call using pyowm.

Hint: search for "forecast_at_place" in the code recipies of the documentation

In [13]:
forecast = mgr.forecast_at_place("London,GB", "3h")
# Get the forecast list
forecast_list = forecast.forecast

for weather in forecast_list:
    print(
        "Time:", weather.reference_time('iso'),
        "Temp:", weather.temperature('celsius')['temp'],
        "Status:", weather.detailed_status
    )

Time: 2026-03-03 12:00:00+00:00 Temp: 10.09 Status: broken clouds
Time: 2026-03-03 15:00:00+00:00 Temp: 12.32 Status: overcast clouds
Time: 2026-03-03 18:00:00+00:00 Temp: 11.12 Status: overcast clouds
Time: 2026-03-03 21:00:00+00:00 Temp: 8.81 Status: broken clouds
Time: 2026-03-04 00:00:00+00:00 Temp: 7.56 Status: broken clouds
Time: 2026-03-04 03:00:00+00:00 Temp: 6.88 Status: overcast clouds
Time: 2026-03-04 06:00:00+00:00 Temp: 6.33 Status: broken clouds
Time: 2026-03-04 09:00:00+00:00 Temp: 8.87 Status: broken clouds
Time: 2026-03-04 12:00:00+00:00 Temp: 13.85 Status: broken clouds
Time: 2026-03-04 15:00:00+00:00 Temp: 15.83 Status: clear sky
Time: 2026-03-04 18:00:00+00:00 Temp: 12.65 Status: clear sky
Time: 2026-03-04 21:00:00+00:00 Temp: 10.76 Status: clear sky
Time: 2026-03-05 00:00:00+00:00 Temp: 10.31 Status: clear sky
Time: 2026-03-05 03:00:00+00:00 Temp: 9.06 Status: clear sky
Time: 2026-03-05 06:00:00+00:00 Temp: 8.04 Status: clear sky
Time: 2026-03-05 09:00:00+00:00 Tem

## use case 4: Convenience methods

Wrappers often offer built-in methods to handle common kinds of tasks related to the APIs, reducing the need for manual calculations.

for example converting units (e.g., temperature from Celsius to Fahrenheit) or working with more complex data requires manual conversion when using the raw API.

In [14]:
city = 'London'
url = f'http://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}'

response = requests.get(url)
data = response.json()
temperature_celsius = data['main']['temp']
temperature_fahrenheit = (temperature_celsius * 9/5) + 32

print(f"Temperature in Celsius: {temperature_celsius}°C, Fahrenheit: {temperature_fahrenheit}°F")

Temperature in Celsius: 282.34°C, Fahrenheit: 540.212°F


But the pyowm wrapper offers built-in methods to handle these kinds of tasks, reducing the need for manual calculations.
Get the temperature both in Celcius and Farenheit using pyowm. Navigate the code recipes to figure out the inbuilt methods for this.

In [23]:
mgr = owm.weather_manager()

forecaster = mgr.forecast_at_place("London,GB", "3h")  # Forecaster object
forecast = forecaster.forecast                         # Forecast object

weather = forecast.weathers[0]  # first Weather object in the forecast

temp_c = weather.temperature("celsius")["temp"]
temp_f = weather.temperature("fahrenheit")["temp"]

print("Time:", weather.reference_time("iso"))
print("Temp (C):", temp_c)
print("Temp (F):", temp_f)

Time: 2026-03-03 12:00:00+00:00
Temp (C): 10.28
Temp (F): 50.5
